# Track 7 · Lesson 1 — Value Iteration & Q-learning

Companion notebook (Track 7 · RL). Dynamic programming with the model, then model-free Q-learning. Only numpy needed.

In [ ]:
"""
Track 7 · Lesson 1 — GridWorld MDP + Value Iteration (dynamic programming)
Run:  python gridworld.py

A Markov Decision Process is (States, Actions, Transitions P, Rewards R, discount gamma).
Here: a 4x4 grid. The agent starts top-left, wants the goal (+1) bottom-right, and
must avoid a pit (-1). Moves are "slippery": with prob 0.8 you go where you intend,
else you slip sideways. Value Iteration computes the optimal value of every state by
repeatedly applying the Bellman optimality backup until it stops changing.
"""
import numpy as np

ROWS, COLS = 4, 4
GOAL, PIT, WALL = (3, 3), (1, 3), (1, 1)
GAMMA = 0.9
ACTIONS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
A_LIST = list(ACTIONS)

def in_grid(r, c):
    return 0 <= r < ROWS and 0 <= c < COLS and (r, c) != WALL

def reward(s):
    """Reward for ENTERING state s."""
    if s == GOAL: return 1.0
    if s == PIT:  return -1.0
    return -0.02                                   # small step cost: hurry up

def is_terminal(s):
    return s == GOAL or s == PIT

def transitions(s, a):
    """Slippery: 0.8 intended, 0.1 each perpendicular. Returns list of (prob, s')."""
    if is_terminal(s): return [(1.0, s)]
    intended = ACTIONS[a]
    perp = [(-intended[1], intended[0]), (intended[1], -intended[0])]
    out = []
    for prob, (dr, dc) in [(0.8, intended), (0.1, perp[0]), (0.1, perp[1])]:
        nr, nc = s[0] + dr, s[1] + dc
        ns = (nr, nc) if in_grid(nr, nc) else s     # bump into wall/edge -> stay
        out.append((prob, ns))
    return out

def value_iteration(theta=1e-6):
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS) if (r, c) != WALL}
    while True:
        delta = 0.0
        for s in V:
            if is_terminal(s):
                V[s] = 0.0; continue            # terminal: no future reward
            best = max(sum(p * (reward(ns) + GAMMA * (0.0 if is_terminal(ns) else V[ns]))
                           for p, ns in transitions(s, a)) for a in A_LIST)
            delta = max(delta, abs(best - V[s])); V[s] = best
        if delta < theta: break
    # greedy policy from V
    policy = {}
    for s in V:
        if is_terminal(s): policy[s] = None; continue
        policy[s] = max(A_LIST, key=lambda a: sum(p * (reward(ns) + GAMMA * (0.0 if is_terminal(ns) else V[ns]))
                                                   for p, ns in transitions(s, a)))
    return V, policy

def main():
    V, policy = value_iteration()
    print("Optimal state values (Value Iteration):")
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            s = (r, c)
            row.append(" WALL " if s == WALL else f"{V[s]:+.2f}")
        print("  ".join(row))
    print("\nOptimal policy (arrows):")
    arrows = {"up": "^", "down": "v", "left": "<", "right": ">"}
    for r in range(ROWS):
        line = []
        for c in range(COLS):
            s = (r, c)
            if s == WALL: line.append("#")
            elif s == GOAL: line.append("G")
            elif s == PIT: line.append("X")
            else: line.append(arrows[policy[s]])
        print(" ".join(line))
    return V, policy


"""
Track 7 · Lesson 1 — Q-learning (model-free control) on the GridWorld
Run:  python q_learning.py

Value Iteration needed the full model (P and R). Q-learning does NOT: it learns
purely from experience via the temporal-difference update
    Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ].
We explore with a decaying epsilon-greedy policy and use "exploring starts" (each
episode begins in a random state) so every state gets visited and learned.
"""
import numpy as np


rng = np.random.default_rng(0)
STATES = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) != WALL]
NONTERM = [s for s in STATES if not is_terminal(s)]

def step(s, a):
    """Sample a next state from the (slippery) environment, and the reward of s."""
    probs = [p for p, _ in transitions(s, a)]
    nss   = [ns for _, ns in transitions(s, a)]
    ns = nss[rng.choice(len(nss), p=probs)]
    return ns, reward(ns)   # reward for ENTERING ns

def greedy_return(Q, start=(0, 0), rollouts=200):
    """Average return of the current greedy policy from `start` (evaluation only)."""
    tot = 0.0
    for _ in range(rollouts):
        s = start; g = 0.0
        for t in range(100):
            if is_terminal(s): break
            a = max(A_LIST, key=lambda a: Q[(s, a)])
            s, r = step(s, a); g += r
        tot += g
    return tot / rollouts

def q_learning(episodes=8000, alpha=0.2):
    Q = {(s, a): 0.0 for s in STATES for a in A_LIST}
    curve = []
    for ep in range(episodes):
        eps = max(0.02, 0.3 * (1 - ep / episodes))      # decay exploration
        s = NONTERM[rng.integers(len(NONTERM))]         # exploring start
        for t in range(100):
            if is_terminal(s): break
            a = rng.choice(A_LIST) if rng.random() < eps else \
                max(A_LIST, key=lambda a: Q[(s, a)])
            ns, r = step(s, a)
            best_next = 0.0 if is_terminal(ns) else max(Q[(ns, a2)] for a2 in A_LIST)
            Q[(s, a)] += alpha * (r + GAMMA * best_next - Q[(s, a)])   # TD update
            s = ns
        if ep % 100 == 0:
            curve.append((ep, greedy_return(Q)))
    return Q, curve

def policy_grid(Q):
    arrows = {"up": "^", "down": "v", "left": "<", "right": ">"}
    rows = []
    for r in range(ROWS):
        line = []
        for c in range(COLS):
            s = (r, c)
            if s == WALL: line.append("#")
            elif s == GOAL: line.append("G")
            elif s == PIT: line.append("X")
            else: line.append(arrows[max(A_LIST, key=lambda a: Q[(s, a)])])
        rows.append(" ".join(line))
    return "\n".join(rows)

def main():
    Q, curve = q_learning()
    print(f"Trained Q-learning ({len(curve)*100} episodes, exploring starts).")
    print("\nLearned policy (greedy w.r.t. Q):")
    print(policy_grid(Q))
    print(f"\nGreedy return from (0,0): {greedy_return(Q):+.3f}  (undiscounted; ~+0.86 is near-optimal)")
    return Q, curve


# ---- run it ----
V, pol = value_iteration()
print("Value Iteration — optimal values:")
for r in range(ROWS):
    print("  ".join("WALL" if (r,c)==WALL else f"{V[(r,c)]:+.2f}" for c in range(COLS)))
print("\nQ-learning (model-free):")
main()
